In [2]:
import os
import numpy as np
from natsort import natsorted
import mat73
from scipy import interpolate
from scipy.io import savemat, loadmat
import h5py
import torch
import matplotlib.pyplot as plt
import pickle
from torchvision import transforms
from torchvision.transforms.functional import hflip, vflip

from dflat.render import hsi_to_rgb, general_convolve

def load_mat_dat(path):
    mat_dat = mat73.loadmat(path)
    bands = mat_dat["bands"]
    cube = mat_dat["cube"]
    _ = mat_dat["norm_factor"]
    return bands, cube

def interpolate_HS_Cube(new_channels_nm, hs_cube, hs_bands):
    # Throw an error if we try to extrapolate
    if (min(new_channels_nm) < min(hs_bands) - 1) or (
        max(new_channels_nm) > max(hs_bands) + 1
    ):
        raise ValueError(
            f"In generator, extrapoaltion of the dataset outside of measurement data is not allowed: {min(hs_bands)}-{max(hs_bands)}"
        )

    interpfun = interpolate.interp1d(
        hs_bands,
        hs_cube,
        axis=-1,
        kind="linear",
        assume_sorted=True,
        fill_value="extrapolate",
        bounds_error=False,
    )
    resampled = interpfun(new_channels_nm)

    return resampled

# Resave the raw data to float32 scipy .mat format

In [ ]:
# Resave the original ARAD release as a .mat file compatible with older scipy loadmat since it is faster
# this format does not support float16 but does support float32 which we use throughout instead of float64
dtype = np.float32
root_dir = "/home/deanhazineh/ssdnvme/ARAD1k_Dataset/Arad1k/"
saveto = "./datasets/ARAD1k_repackaged/"

for label in ["valid", "train"]:
    fnames = natsorted(os.listdir(os.path.join(root_dir, label)))
    for it, f in enumerate(fnames):
        try:
            b, cube = load_mat_dat(os.path.join(root_dir, f))
            cube = cube.astype(dtype)
            cube = cube / cube.max()
            mat_dict = {"hsi": cube}
            savemat(os.path.join(saveto, f"ARAD_{label}_{it}.mat"), mat_dict)
        except:
            print(f"file {f} failed to load")

In [3]:
# For CASSI Renderings, we resave the cubes with 256x256 chunking and float16
# We will load these datacubes, grab random 256x256 chunks, render the CASSI measurements, and then train on patches
# Rendering CASSI on the fly is fast since it uses a quick shift, multiply, and sum
datpath = "./datasets/ARAD1k_repackaged/"
savepath = f"./datasets/CASSI_Dataset_450_650/" 
os.makedirs(savepath, exist_ok=True)

fnames = natsorted(os.listdir(datpath))
interp_ch = np.linspace(450, 650, 28)
ch = np.linspace(400, 700, 31)
for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"]
    hsi = hsi / hsi.max()
    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float32)

    hdf5_file_path = savepath+f'{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(256, 256, 28), dtype='float16')


In [7]:
# Save a version of the HSI cubes in a common range of 420 to 700 with 64 chunking (float16)
datpath = "./datasets/ARAD1k_repackaged/"
savepath = "./datasets/HSI_multiset_420_700/"
os.makedirs(savepath, exist_ok=True)

fnames = natsorted(os.listdir(datpath))
ch = np.linspace(400, 700, 31)
interp_ch = np.arange(420, 710, 10)
chunks = 64
dtype='float32'

for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"]
    hsi = hsi / hsi.max()
    hsi = np.clip(interpolate_HS_Cube(interp_ch, hsi, ch),0,1).astype(np.float16)

    hdf5_file_path = savepath+f'{os.path.splitext(f)[0]}.h5'
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(chunks, chunks, 29), dtype=dtype)


# Generate ARAD Training Pair at 256x256 Resolution

In [ ]:
## For the diffvis (metalens) ARAD experiment we want to prerender the data and enable fast patch loading using h5 chunking
## For this version, we just want to use resized 256 HSI images and well draw 64 patches from it
# Change the lens path for each PSF dataset
aug = True
datpath = "./datasets/ARAD1k_repackaged/"
psf_path = "./metasurfaces/L4v2_lens_psf_compact32.pickle"
savepath = "./datasets/metalens_prerendered/prerendered_L4v2_256/"
os.makedirs(savepath, exist_ok=True)

with open(psf_path, "rb") as file:
    data = pickle.load(file)
    psf = data["psf_int"]
    psf = psf / np.max(psf)
    wl = data["wl"]
psf = torch.tensor(psf / np.max(psf), dtype=torch.float32).to(device="cuda")

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize([256,256]),
    ]
)
chunks = 64
dtype='float32'

def render_and_save(hsi, savepath, f, auglabel):
    mhsi = general_convolve(hsi, psf, rfft=True)
    mgs = torch.sum(mhsi, dim=0, keepdim=True)
    mgs = mgs / torch.max(mgs)

    mrgb = hsi_to_rgb(mhsi, wl, tensor_ordering=True, process='raw', projection="Basler_Bayer", normalize=True)
    rgb = hsi_to_rgb(hsi, wl, tensor_ordering=True, process='raw', projection="Basler_Bayer", normalize=True)
    mhsi = mhsi / torch.max(mhsi)
    
    mrgb3 = hsi_to_rgb(mhsi, wl, tensor_ordering=True, process='ideal', projection="Basler_Bayer", normalize=True)
    rgb3 = hsi_to_rgb(hsi, wl, tensor_ordering=True, process='ideal', projection="Basler_Bayer", normalize=True)

    hsi = hsi.permute(1,2,0).cpu().numpy()
    mgs = mgs.permute(1,2,0).cpu().numpy()
    mrgb = mrgb.permute(1,2,0).cpu().numpy()
    rgb = rgb.permute(1,2,0).cpu().numpy()
    mhsi = mhsi.permute(1,2,0).cpu().numpy()
    mrgb3 = mrgb3.permute(1,2,0).cpu().numpy()
    rgb3 = rgb3.permute(1,2,0).cpu().numpy()

    hdf5_file_path = os.path.join(savepath, f'{os.path.splitext(f)[0]}{auglabel}.h5')
    with h5py.File(hdf5_file_path, 'w') as hf:
        hf.create_dataset('hsi', data=hsi, chunks=(chunks, chunks, 31), dtype=dtype)
        hf.create_dataset('mhsi', data=mhsi, chunks=(chunks, chunks, 31), dtype=dtype)
        hf.create_dataset('mgs', data=mgs, chunks=(chunks, chunks, 1), dtype=dtype)
        hf.create_dataset('mrgb', data=mrgb, chunks=(chunks, chunks, 1), dtype=dtype)
        hf.create_dataset('rgb', data=rgb, chunks=(chunks, chunks, 1), dtype=dtype)
        hf.create_dataset('mrgb3', data=mrgb3, chunks=(chunks, chunks, 3), dtype=dtype)
        hf.create_dataset('rgb3', data=rgb3, chunks=(chunks, chunks, 3), dtype=dtype)

    return

fnames = natsorted(os.listdir(datpath))
for f in fnames:
    datfile = os.path.join(datpath, f)
    hsi = loadmat(datfile)["hsi"]
    hsi = transform(hsi).to('cuda')
    hsi = hsi / hsi.max()
    render_and_save(hsi, savepath, f, "")
    if aug: 
        render_and_save(hsi.flip(dims=[2]), savepath, f, "_hflip")
        render_and_save(hsi.flip(dims=[1]), savepath, f, "_vflip")
        render_and_save(hsi.flip(dims=[1,2]), savepath, f, "_vhflip")
